# Stream Processing Fundamentals: Micro-Batch vs Continuous Processing

**Module**: 09_streaming  
**Notebook**: 01_streaming_fundamentals.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The evolution of Spark streaming: legacy DStreams to Structured Streaming, the unbounded table mental model, and latency vs throughput trade-offs.
- **Why does this topic exist?**: Real-time streaming powers anomaly detection, live analytics dashboards, and event-driven architectures.
- **What problem does it solve?**: Managing unbounded streams without state store explosion.
- **Where is it used in real-world Data Engineering?**: Real-time payment fraud scoring and clickstream analysis.

---

## 2. Structured Streaming Mental Model: Stream as an Unbounded Table
```
Incoming Event Stream:
[Event 1] ---> [Event 2] ---> [Event 3] ---> ... (Never ends!)
      |             |              |
      v             v              v
+-----------------------------------------------------------+
| UNBOUNDED INPUT TABLE (New rows appended continuously)    |
+-----------------------------------------------------------+
                            |
                            v Query Expression (DataFrame / SQL)
+-----------------------------------------------------------+
| RESULT TABLE (Updated incrementally per micro-batch)     |
+-----------------------------------------------------------+
                            |
                            v Output Mode (Append / Update / Complete)
+-----------------------------------------------------------+
| EXTERNAL SINK (Console, Files, Kafka, Delta Lake)        |
+-----------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Streaming_Fund_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Rate Stream Source Demo (Synthetic Stream)

In [ ]:
# Create built-in rate source (generates sequential numbers at specified rows/second)
rate_stream = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

print("Is stream active in DataFrame?:", rate_stream.isStreaming)
rate_stream.printSchema()

---

## Experiment: Running a Short-Lived Streaming Query to In-Memory Sink

- **1. What are we testing?**: Start a micro-batch query writing to memory sink and inspect results.
- **2. What do we expect?**: Query populates an in-memory table that can be queried with spark.sql().

In [ ]:
query = (
    rate_stream
    .writeStream
    .format("memory")
    .queryName("rate_sink")
    .outputMode("append")
    .start()
)

# Wait 4 seconds for a few micro-batches to process
time.sleep(4)

print("Querying in-memory streaming sink:")
spark.sql("SELECT * FROM rate_sink ORDER BY timestamp DESC LIMIT 5").show()

query.stop()
print("Streaming query stopped successfully.")

**3. What happened?**: Rate stream generated timestamped integers into in-memory table.  
**4. Why did it happen?**: Structured Streaming micro-batches processed incoming rows as fast as possible.  
**5. Practical Engineering Lesson**: Memory sink is ideal for unit testing and interactive debugging in Jupyter.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Stream Status
**Problem Statement**: Query query.isActive and query.status.  
**Input / Context**: query

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Rate Source with Ramptime
**Problem Statement**: Configure rampUpTime option on rate source.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Explain Legacy DStreams
**Problem Statement**: Contrast DStreams with Structured Streaming.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Stream Status):
```python
print('query.isActive, query.status')
```

#### Solution 2 (Rate Source with Ramptime):
```python
spark.readStream.format('rate').option('rampUpTime','5s').load()
```

#### Solution 3 (Explain Legacy DStreams):
```python
print('DStreams used RDDs; Structured Streaming uses the Catalyst DataFrame engine with native event-time.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Unbounded Table**: Stream is conceptually an infinite append-only table.
- **Micro-batch**: Processes incoming records in small discrete batches.

### Essential Technical Terms
- `isStreaming`: Boolean flag indicating streaming DataFrame.
- `rate source`: Synthetic benchmark source.

### Common Engineering Mistakes
- **Calling collect() on streaming DataFrame**: Throws AnalysisException; streaming data must use writeStream.

### Interview Questions on This Topic
- **Q: What is Structured Streaming?**
  - *A*: A scalable stream processing engine built on the Spark SQL engine, treating streams as unbounded tables.

### What to Study Next
Open 02_structured_streaming.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')